# Simulated Annealing for TSP

## Lab 4 – Work during the lab

**Tasks:**
1. Implement Simulated Annealing for TSP
2. Test on the same two instances as Lab 3: `pr107` (group A) and `zi929` (group B), with different parameter settings

**Instances:** `../tsp_instances/A/pr107.tsp` (107 cities), `../tsp_instances/B/zi929.tsp` (929 cities)

### Imports and global variables

In [1]:
import math
import os
import random
import time

### Helper functions

Reused from Lab 3: TSP file parser, distance matrix builder, tour cost, greedy initialisation, and O(1) 2-opt delta evaluation.

In [2]:
# ── TSP I/O ───────────────────────────────────────────────────────────────────

def load_tsp(file_name: str) -> list[tuple[float, float]]:
    """
    Parses a TSPLIB EUC_2D file.
    Returns a list of (x, y) city coordinates.
    """
    coords = []
    if not os.path.exists(file_name):
        print(f"Error: File '{file_name}' not found.")
        return coords

    in_node_section = False
    with open(file_name) as f:
        for line in f:
            line = line.strip()
            if line.startswith("NODE_COORD_SECTION"):
                in_node_section = True
                continue
            if line in ("EOF", ""):
                in_node_section = False
                continue
            if in_node_section:
                parts = line.split()
                if len(parts) >= 3:
                    coords.append((float(parts[1]), float(parts[2])))
    return coords


def build_distance_matrix(coords: list[tuple[float, float]]) -> list[list[float]]:
    """Pre-computes all pairwise Euclidean distances."""
    n = len(coords)
    dist = [[0.0] * n for _ in range(n)]
    for i in range(n):
        for j in range(i + 1, n):
            dx = coords[i][0] - coords[j][0]
            dy = coords[i][1] - coords[j][1]
            d = math.sqrt(dx * dx + dy * dy)
            dist[i][j] = d
            dist[j][i] = d
    return dist


def tour_cost(tour: list[int], dist: list[list[float]]) -> float:
    """Total cost of a closed TSP tour."""
    n = len(tour)
    return sum(dist[tour[i]][tour[(i + 1) % n]] for i in range(n))


# ── Greedy initialisation ─────────────────────────────────────────────────────

def greedy_tsp(dist: list[list[float]], start: int = 0) -> list[int]:
    """
    Nearest-neighbour greedy heuristic.
    Returns a tour (list of city indices).
    """
    n = len(dist)
    visited = [False] * n
    tour = [start]
    visited[start] = True
    for _ in range(n - 1):
        current = tour[-1]
        nearest = min(
            (j for j in range(n) if not visited[j]),
            key=lambda j: dist[current][j]
        )
        tour.append(nearest)
        visited[nearest] = True
    return tour


def verify_tour(tour: list[int], n: int) -> bool:
    """Checks that a tour is a valid permutation of all city indices."""
    return sorted(tour) == list(range(n))


# ── 2-opt neighbourhood ───────────────────────────────────────────────────────

def apply_2opt_move(tour: list[int], i: int, j: int) -> list[int]:
    """
    Applies a 2-opt move by reversing the segment tour[i+1 : j+1].
    Returns a new tour without modifying the original.
    """
    return tour[:i + 1] + tour[i + 1:j + 1][::-1] + tour[j + 1:]


def two_opt_delta(tour: list[int], dist: list[list[float]], i: int, j: int) -> float:
    """
    O(1) cost change of reversing tour[i+1 : j+1].
    Removes edges (tour[i]→tour[i+1]) and (tour[j]→tour[j+1 % n]).
    Adds    edges (tour[i]→tour[j])   and (tour[i+1]→tour[j+1 % n]).
    """
    n = len(tour)
    a, b = tour[i],     tour[i + 1]
    c, d = tour[j],     tour[(j + 1) % n]
    return dist[a][c] + dist[b][d] - dist[a][b] - dist[c][d]

### Core Algorithm

**Simulated Annealing (SA)** is a probabilistic local search metaheuristic inspired by the annealing process in metallurgy. Starting from a high temperature T, the algorithm progressively lowers T according to a *cooling schedule* g. At each step a random neighbour x is drawn; if x improves the current solution it is always accepted, otherwise it is accepted with probability e^((cost(c)−cost(x))/T). As T → 0 the acceptance of worsening moves becomes increasingly rare, focusing the search.

**Cooling schedules** (as per lab specification):
- **Geometric:** `T(k+1) = alpha * T(k)`, alpha ∈ (0,1) close to 1 — e.g. 0.999, 0.99, 0.95
- **Logarithmic:** `T(k+1) = T(k) / log(k+2)`
- **Linear:** `T(k+1) = T(k) / (k+1)`

**Neighbourhood:** a single random 2-opt move (reverse a random sub-segment of the tour).

**Pseudocode (from lab sheet):**
```
t = 0
initialise T
c = greedy_solution()          # warm start
evaluate c
best = c

repeat                         # outer loop: one temperature level per iteration
    repeat                     # inner loop: INNER_ITERS trials at current T
        x = random 2-opt neighbour of c
        if cost(x) < cost(c):
            c ← x
        else if random[0,1) < exp((cost(c) − cost(x)) / T):
            c ← x              # accept worsening move
        if cost(c) < cost(best):
            best ← c
    until (inner termination condition)
    T ← g(T, t)
    t ← t + 1
until (T < T_min)              # halting criterion

return best
```

**Key design choices:**
- **Warm start:** greedy nearest-neighbour tour from a random city (introduces run-to-run variance)
- **Neighbourhood:** single uniformly random 2-opt move per inner iteration — O(1) with delta eval
- **Halting:** outer loop stops when T < T_min

In [3]:
def simulated_annealing_tsp(
    dist: list[list[float]],
    initial_temp: float = 10000.0,
    min_temp: float = 0.1,
    alpha: float = 0.99,
    inner_iters: int = 100,
    cooling: str = "geometric",
) -> dict:
    """
    Simulated Annealing for TSP using 2-opt moves.

    Parameters:
        dist         : distance matrix (n x n)
        initial_temp : starting temperature T0
        min_temp     : halting criterion — stop when T < min_temp
        alpha        : cooling rate for geometric schedule (ignored otherwise)
        inner_iters  : number of random-move trials per temperature level
        cooling      : 'geometric' | 'log' | 'linear'

    Returns a dict with keys: 'tour', 'cost', 'iterations'
    """
    n = len(dist)

    # Warm start: greedy tour from a random city (gives run-to-run variance)
    start_city = random.randint(0, n - 1)
    current = greedy_tsp(dist, start=start_city)
    current_cost = tour_cost(current, dist)

    best = current[:]
    best_cost = current_cost

    T = initial_temp
    t = 0  # outer iteration counter (temperature level)

    while T > min_temp:
        # ── Inner loop: INNER_ITERS trials at the current temperature ──
        for _ in range(inner_iters):
            # Select a random 2-opt move
            i = random.randint(0, n - 3)
            j = random.randint(i + 2, n - 1)

            delta = two_opt_delta(current, dist, i, j)

            # Accept if improving, or probabilistically if worsening
            if delta < 0:
                current = apply_2opt_move(current, i, j)
                current_cost += delta
            elif random.random() < math.exp(-delta / T):
                current = apply_2opt_move(current, i, j)
                current_cost += delta

            # Track global best
            if current_cost < best_cost:
                best = current[:]
                best_cost = current_cost

        # ── Cool the temperature ──
        t += 1
        if cooling == "geometric":
            T *= alpha
        elif cooling == "log":
            T = initial_temp / math.log(t + 2)   # log(k+1) with k=t+1 => log(t+2)
        elif cooling == "linear":
            T = initial_temp / (t + 1)            # T(k)/k with k=t+1

    return {
        "tour": best,
        "cost": best_cost,
        "iterations": t,
    }

### Experiments

We test three cooling schedules and three alpha values (for geometric cooling) on both instances.
Each configuration is run `NUM_RUNS` times; we report the best cost, average cost, and average run time.
Because SA is stochastic (greedy start from a random city), multiple runs produce genuinely different results.

In [4]:
# Parameter grid
TSP_INSTANCES  = [
    {"name": "../tsp_instances/A/pr107.tsp", "label": "pr107"},
    {"name": "../tsp_instances/B/zi929.tsp",  "label": "zi929"},
]

INITIAL_TEMP = 10000.0
MIN_TEMP     = 0.1
INNER_ITERS  = 100
NUM_RUNS     = 5
OUTPUT_FILE  = "sa_tsp_results.md"

# Configurations: (cooling_schedule, alpha_label, alpha_value)
CONFIGS = [
    ("geometric", "alpha=0.999", 0.999),
    ("geometric", "alpha=0.99",  0.99),
    ("geometric", "alpha=0.95",  0.95)
]

In [5]:
def run_sa_experiments(num_runs: int = NUM_RUNS) -> None:
    """
    Runs SA for each (instance, cooling config) combination.
    Prints a summary and saves results to OUTPUT_FILE.
    """
    with open(OUTPUT_FILE, "w") as f:
        f.write("# Simulated Annealing for TSP – Results\n\n")
        f.write("## Configuration\n")
        f.write(f"- **Instances:** pr107 (n=107, group A), zi929 (n=929, group B)\n")
        f.write(f"- **T0:** {INITIAL_TEMP}  |  **T_min:** {MIN_TEMP}  |  **Inner iters:** {INNER_ITERS}\n")
        f.write(f"- **Runs per configuration:** {num_runs}\n")
        f.write(f"- **Cooling schedules tested:** geometric (alpha=0.999/0.99/0.95), log, linear\n\n")
        f.write("## Results\n\n")
        f.write(
            f"| {'Instance':<10} | {'Cooling':<12} | {'Best Cost':<12} "
            f"| {'Avg Cost':<12} | {'Avg Time (s)':<13} | {'Outer Iters':<12} |\n"
        )
        f.write("|---|---|---|---|---|---|\n")

    for inst in TSP_INSTANCES:
        coords = load_tsp(inst["name"])
        if not coords:
            continue

        label    = inst["label"]
        n_cities = len(coords)
        dist_mat = build_distance_matrix(coords)

        # Greedy baseline (fixed start for reproducibility)
        greedy_tour = greedy_tsp(dist_mat, start=0)
        greedy_cost = tour_cost(greedy_tour, dist_mat)

        print(f"\n{'='*60}")
        print(f"Instance: {label} (n={n_cities})  |  greedy baseline: {greedy_cost:.2f}")
        print(f"{'='*60}")

        for cooling, cfg_label, alpha in CONFIGS:
            best_cost  = float("inf")
            sum_costs  = 0.0
            total_time = 0.0
            total_iters = 0

            for _ in range(num_runs):
                t0 = time.time()
                result = simulated_annealing_tsp(
                    dist_mat,
                    initial_temp=INITIAL_TEMP,
                    min_temp=MIN_TEMP,
                    alpha=alpha if alpha else 0.99,  # alpha unused for log/linear
                    inner_iters=INNER_ITERS,
                    cooling=cooling,
                )
                elapsed = time.time() - t0

                total_time  += elapsed
                sum_costs   += result["cost"]
                total_iters += result["iterations"]
                if result["cost"] < best_cost:
                    best_cost = result["cost"]

            avg_cost  = sum_costs   / num_runs
            avg_time  = total_time  / num_runs
            avg_iters = total_iters / num_runs

            print(
                f"  {cfg_label:<12} | best={best_cost:10.2f}  "
                f"avg={avg_cost:10.2f}  time={avg_time:.3f}s  iters={avg_iters:.0f}"
            )

            with open(OUTPUT_FILE, "a") as f:
                f.write(
                    f"| {label:<10} | {cfg_label:<12} | {best_cost:<12.2f} "
                    f"| {avg_cost:<12.2f} | {avg_time:<13.4f} | {avg_iters:<12.0f} |\n"
                )

        # Write greedy baseline row
        with open(OUTPUT_FILE, "a") as f:
            f.write(
                f"| {label:<10} | {'greedy':<12} | {greedy_cost:<12.2f} "
                f"| {greedy_cost:<12.2f} | {'0.0000':<13} | {'0':<12} |\n"
            )

    print(f"\nResults saved to {OUTPUT_FILE}")


if __name__ == "__main__":
    run_sa_experiments(NUM_RUNS)


Instance: pr107 (n=107)  |  greedy baseline: 46678.15
  alpha=0.999  | best=  44491.08  avg=  44701.54  time=1.614s  iters=11508
  alpha=0.99   | best=  44947.90  avg=  45193.54  time=0.160s  iters=1146
  alpha=0.95   | best=  46026.49  avg=  47612.23  time=0.033s  iters=225

Instance: zi929 (n=929)  |  greedy baseline: 117733.70
  alpha=0.999  | best= 113979.02  avg= 116352.18  time=3.883s  iters=11508
  alpha=0.99   | best= 116403.60  avg= 118702.41  time=0.490s  iters=1146
  alpha=0.95   | best= 112984.00  avg= 118930.79  time=0.153s  iters=225

Results saved to sa_tsp_results.md
